In [59]:
import pandas as pd
import os

In [60]:
def parse_csv_manual(text):
    text = text.strip()
    if text.startswith('"') and text.endswith('"'):
        text = text[1:-1]
        text = text.replace('""', '"')

    columns = []
    current_col = ""
    inside_quotes = False

    i = 0
    while i < len(text):
        char = text[i]

        if char == '"':
            if inside_quotes and i + 1 < len(text) and text[i + 1] == '"':
                current_col += '"'
                i += 2
                continue
            else:
                inside_quotes = not inside_quotes
                i += 1
                continue

        if char == ',' and not inside_quotes:
            columns.append(current_col)
            current_col = ""
            i += 1
            continue

        current_col += char
        i += 1

    columns.append(current_col)
    return columns


In [61]:
def load_csv_to_df(filename, role):
    """Load CSV dan return DataFrame dengan role"""
    with open(filename, 'r', encoding='utf-8-sig') as f:
        content = f.read()

    lines = content.strip().split('\n')
    header = parse_csv_manual(lines[0])

    data = []
    for i, line in enumerate(lines[1:], start=1):
        if not line.strip():
            continue
        row = parse_csv_manual(line)
        if len(row) == len(header):
            data.append(row)

    df = pd.DataFrame(data, columns=header)

    df = df[df['question_id'].str.isnumeric()]

    df = df.drop(columns=['is_image'], errors='ignore')
    df.insert(0, 'role', role)
    df['is_image'] = 0

    return df

In [62]:
def generate_sql_inserts_compact(df):
    """Generate SQL INSERT compact"""
    generated_questions = {}
    question_counter = 1
    questions_data = []
    options_data = []

    for idx, row in df.iterrows():
        role = row['role']
        q_id = row['question_id']
        q_text = row['question_text'].replace("'", "\\'").replace('"', '\\"')
        opt_letter = row['option_letter']
        opt_text = row['option_text'].replace("'", "\\'").replace('"', '\\"')
        score = int(row['score'])
        is_image = int(row['is_image'])

        question_key = (role, q_id, q_text)

        if question_key not in generated_questions:
            questions_data.append(f"('{role}', '{q_text}')")
            generated_questions[question_key] = question_counter
            db_question_id = question_counter
            question_counter += 1
        else:
            db_question_id = generated_questions[question_key]

        options_data.append(f"({db_question_id}, '{opt_letter}', '{opt_text}', {score}, {is_image})")

    sql_questions = "INSERT INTO `questions` (`role`, `question_text`) VALUES\n"
    sql_questions += ",\n".join(questions_data) + ";"

    sql_options = "INSERT INTO `options` (`question_id`, `option_letter`, `option_text`, `score`, `is_image`) VALUES\n"
    sql_options += ",\n".join(options_data) + ";"

    return sql_questions, sql_options

In [63]:
files_mapping = {
    '/content/_kkm_questions.csv': 'kkm',
    '/content/_masinis_2_questions.csv': 'masinis_2',
    '/content/_masinis_3_questions.csv': 'masinis_3',
    '/content/_mualim_1_questions.csv': 'mualim_1',
    '/content/_mualim_2_questions.csv': 'mualim_2',
    '/content/_nahkoda_questions.csv': 'nahkoda',
    '/content/_va_1_questions.csv': 'va_1',
    '/content/_va_2_questions.csv': 'va_2',
    '/content/_va_3_questions.csv': 'va_3'
}

all_dfs = []
for filepath, role in files_mapping.items():
    if os.path.exists(filepath):
        print(f"Processing: {role}...")
        df = load_csv_to_df(filepath, role)
        all_dfs.append(df)
        print(f"Loaded {len(df)} rows")
    else:
        print(f"File not found: {filepath}")

# Gabung semua DF
combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"\n=== TOTAL ===")
print(f"Total rows: {len(combined_df)}")
print(f"Total unique questions: {len(combined_df.drop_duplicates(['role', 'question_id', 'question_text']))}")


Processing: kkm...
Loaded 200 rows
Processing: masinis_2...
Loaded 197 rows
Processing: masinis_3...
Loaded 200 rows
Processing: mualim_1...
Loaded 200 rows
Processing: mualim_2...
Loaded 200 rows
Processing: nahkoda...
Loaded 200 rows
Processing: va_1...
Loaded 120 rows
Processing: va_2...
Loaded 120 rows
Processing: va_3...
Loaded 48 rows

=== TOTAL ===
Total rows: 1485
Total unique questions: 412


In [65]:
# Generate SQL
sql_questions, sql_options = generate_sql_inserts_compact(combined_df)

# Save SQL file
with open('insert_all_questions.sql', 'w', encoding='utf-8') as f:
    f.write(sql_questions)
    f.write(sql_options)

# Save TXT file
with open('insert_all_questions.txt', 'w', encoding='utf-8') as f:
    f.write(sql_questions)
    f.write(sql_options)
